# Talent Fit Scoring Pipeline
Rank candidates by fit score using job title similarity and connection count.

In [1]:
from pathlib import Path
from config import DATA_FILE
from src.data_loader import load_data
from src.preprocessing import preprocess
from src.feature_engineering import add_features
from src.ranking import rank_candidates
from src.reranking import rerank
from src.evaluation import ndcg_at_k

ROOT = Path().resolve()
df = load_data(ROOT / DATA_FILE)
df.head()

,id,job_title,location,connection,fit
0,1,2019 C.T. Bauer College of Business Graduate (...,"Houston, Texas",85,NaN
1,2,Native English Teacher at EPIK (English Progra...,Kanada,500+,NaN
2,3,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,NaN
3,4,People Development Coordinator at Ryan,"Denton, Texas",500+,NaN
4,5,Advisory Board Member at Celal Bayar University,"İzmir, Türkiye",500+,NaN


## Inspect raw data

In [2]:
print(df.shape)
print()
df.info()
print()
print(df["location"].value_counts())
print()
print(f"Duplicate rows: {df.duplicated().sum()}")

(104, 5)

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 104 entries, 0 to 103
Data columns (total 5 columns):
 #   Column      Non-Null Count  Dtype  
---  ------      --------------  -----  
 0   id          104 non-null    int64  
 1   job_title   104 non-null    object 
 2   location    104 non-null    object 
 3   connection  104 non-null    object 
 4   fit         0 non-null      float64
dtypes: float64(1), int64(1), object(3)
memory usage: 4.2+ KB

location
Kanada                                 12
Raleigh-Durham, North Carolina Area     8
Houston, Texas Area                     8
Greater New York City Area              7
Houston, Texas                          7
Denton, Texas                           6
Greater Philadelphia Area               5
San Francisco Bay Area                  5
İzmir, Türkiye                          4
Lake Forest, California                 4
Atlanta, Georgia                        4
Chicago, Illinois                       2
Austin, Texas Area     

## Preprocess

In [3]:
df = preprocess(df)
df[["job_title", "job_title_clean", "connections_raw", "connections_norm"]].head(10)

,job_title,job_title_clean,connections_raw,connections_norm
0,2019 C.T. Bauer College of Business Graduate (...,bauer college business graduate magna cum laud...,85,0.170
1,Native English Teacher at EPIK (English Progra...,native english teacher epik english program korea,500,1.000
2,Aspiring Human Resources Professional,aspiring human resource professional,44,0.088
3,People Development Coordinator at Ryan,people development coordinator ryan,500,1.000
4,Advisory Board Member at Celal Bayar University,advisory board member celal bayar university,500,1.000
5,Aspiring Human Resources Specialist,aspiring human resource specialist,1,0.002
6,Student at Humber College and Aspiring Human R...,student humber college aspiring human resource...,61,0.122
7,HR Senior Specialist,senior specialist,500,1.000
8,Student at Humber College and Aspiring Human R...,student humber college aspiring human resource...,61,0.122
9,Seeking Human Resources HRIS and Generalist Po...,seeking human resource hris generalist position,500,1.000


## Feature engineering (TF-IDF cosine similarity)

In [4]:
df = add_features(df)
df[["job_title_clean", "fit"]].sort_values("fit", ascending=False).head(10)

,job_title_clean,fit
98,seeking human resource position,0.343950
27,seeking human resource opportunity,0.340965
29,seeking human resource opportunity,0.340965
2,aspiring human resource professional,0.274372
16,aspiring human resource professional,0.274372
32,aspiring human resource professional,0.274372
45,aspiring human resource professional,0.274372
20,aspiring human resource professional,0.274372
57,aspiring human resource professional,0.274372
96,aspiring human resource professional,0.274372


## Rank candidates

In [5]:
ranked = rank_candidates(df)
ranked[["id", "job_title", "location", "connections_raw", "fit"]].head(10)

,id,job_title,location,connections_raw,fit
0,40,Seeking Human Resources HRIS and Generalist Po...,Greater Philadelphia Area,500,0.474526
1,10,Seeking Human Resources HRIS and Generalist Po...,Greater Philadelphia Area,500,0.474526
2,62,Seeking Human Resources HRIS and Generalist Po...,Greater Philadelphia Area,500,0.474526
3,53,Seeking Human Resources HRIS and Generalist Po...,Greater Philadelphia Area,500,0.474526
4,30,Seeking Human Resources Opportunities,"Chicago, Illinois",390,0.472676
5,28,Seeking Human Resources Opportunities,"Chicago, Illinois",390,0.472676
6,75,"Nortia Staffing is seeking Human Resources, Pa...","San Jose, California",500,0.417223
7,27,Aspiring Human Resources Management student se...,"Houston, Texas Area",500,0.387228
8,29,Aspiring Human Resources Management student se...,"Houston, Texas Area",500,0.387228
9,94,Seeking Human Resources Opportunities. Open t...,Amerika Birleşik Devletleri,415,0.377511


## Re-rank after human feedback

In [6]:
STARRED_IDS = [1, 3, 6]

reranked = rerank(ranked, STARRED_IDS)
reranked[["id", "job_title", "location", "connections_raw", "fit"]].head(10)

,id,job_title,location,connections_raw,fit
0,97,Aspiring Human Resources Professional,"Kokomo, Indiana Area",71,0.424505
1,33,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,0.416405
2,46,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,0.416405
3,58,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,0.416405
4,3,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,0.416405
5,21,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,0.416405
6,17,Aspiring Human Resources Professional,"Raleigh-Durham, North Carolina Area",44,0.416405
7,6,Aspiring Human Resources Specialist,Greater New York City Area,1,0.356582
8,49,Aspiring Human Resources Specialist,Greater New York City Area,1,0.356582
9,60,Aspiring Human Resources Specialist,Greater New York City Area,1,0.356582


## Evaluate

In [7]:
print(f"NDCG@10 before re-ranking: {ndcg_at_k(ranked,   STARRED_IDS):.4f}")
print(f"NDCG@10 after  re-ranking: {ndcg_at_k(reranked, STARRED_IDS):.4f}")

NDCG@10 before re-ranking: 0.0000
NDCG@10 after  re-ranking: 0.3296
